# Generators and Iterators

## Overview
Iteration is one of Python's most fundamental operations. Understanding the iteration protocol from the ground up reveals why generators are so powerful: they let you produce values **lazily** — one at a time, only when asked — rather than computing everything upfront and storing it in memory.

This matters enormously at scale: a generator can process a 100 GB log file using only kilobytes of memory.

## Table of Contents
1. The Iteration Protocol (`__iter__` / `__next__`)
2. Custom Iterator Class
3. Generator Functions and `yield`
4. Generator Expressions
5. Lazy Evaluation: Memory Comparison
6. Two-Way Communication: `send()` to Generator
7. `yield from` — Delegation
8. The `itertools` Module
9. Common Mistakes
10. Exercises
11. Mini-Project: File Line Reader

## 1. The Iteration Protocol

When Python executes `for x in obj:`, it calls `iter(obj)` which calls `obj.__iter__()` to get an **iterator**. Then it repeatedly calls `next(iterator)` which calls `iterator.__next__()`. When `StopIteration` is raised, the loop ends.

An **iterable** has `__iter__`. An **iterator** has both `__iter__` and `__next__`.

In [ ]:
# Peeking under the hood of a for loop
numbers = [1, 2, 3]

# What `for x in numbers` actually does:
iterator = iter(numbers)     # calls numbers.__iter__()
print(next(iterator))        # calls iterator.__next__() -> 1
print(next(iterator))        # -> 2
print(next(iterator))        # -> 3

try:
    print(next(iterator))    # -> raises StopIteration
except StopIteration:
    print("StopIteration raised — loop ends")

# A list is an ITERABLE (has __iter__)
# iter(list) returns a list_iterator object (which is an ITERATOR)
print(type(numbers))         # <class 'list'>
print(type(iter(numbers)))   # <class 'list_iterator'>

## 2. Custom Iterator Class

By implementing `__iter__` and `__next__`, you can make any object iterable. This is verbose — generators exist to simplify this — but understanding the class-based approach first is essential.

In [ ]:
class CountUp:
    """An iterator that counts from start to stop (inclusive)."""

    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        # An iterator's __iter__ returns itself
        return self

    def __next__(self):
        if self.current > self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

counter = CountUp(1, 5)
for n in counter:
    print(n, end=" ")    # 1 2 3 4 5

print()
# Can also use next() directly
counter2 = CountUp(10, 13)
print(next(counter2))    # 10
print(next(counter2))    # 11

# A class that is ITERABLE but not an ITERATOR
# (creates a fresh iterator each time iter() is called)
class FibSequence:
    """Iterable but not iterator — can be iterated multiple times."""
    def __init__(self, count):
        self.count = count

    def __iter__(self):
        a, b = 0, 1
        for _ in range(self.count):
            yield a        # using yield here makes __iter__ a generator func!
            a, b = b, a + b

fibs = FibSequence(8)
print(list(fibs))    # [0, 1, 1, 2, 3, 5, 8, 13]
print(list(fibs))    # can iterate again — [0, 1, 1, 2, 3, 5, 8, 13]

## 3. Generator Functions and `yield`

A **generator function** contains `yield`. When called, it returns a **generator object** (an iterator). The function body doesn't execute until you call `next()` on it.

Each `yield` **suspends** the function, saving its entire state (local variables, position in code). The next `next()` call **resumes** from exactly where it left off.

In [ ]:
def count_up(start, stop):
    """Generator equivalent of the CountUp class above — much cleaner!"""
    current = start
    while current <= stop:
        yield current   # suspend here, return current
        current += 1    # resumes here on next next() call

gen = count_up(1, 5)
print(type(gen))         # <class 'generator'>
print(next(gen))         # 1 — function runs until yield
print(next(gen))         # 2 — resumes after yield

print(list(count_up(1, 5)))  # [1, 2, 3, 4, 5]

In [ ]:
# Generators can be infinite — impossible with a list!
def integers_from(n):
    """Generates all integers starting from n. Never ends."""
    while True:
        yield n
        n += 1

def take(n, iterable):
    """Take first n elements from iterable."""
    for i, val in enumerate(iterable):
        if i >= n:
            break
        yield val

print(list(take(5, integers_from(100))))  # [100, 101, 102, 103, 104]

# Fibonacci — infinite
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print(first_10)  # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## 4. Generator Expressions

Generator expressions are to generators what list comprehensions are to lists. Use `()` instead of `[]`. They are lazy — values computed on demand.

In [ ]:
# List comprehension: all values computed and stored NOW
squares_list = [x**2 for x in range(10)]

# Generator expression: values computed LAZILY on demand
squares_gen = (x**2 for x in range(10))

print(squares_list)   # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
print(squares_gen)    # <generator object <genexpr> at 0x...>
print(list(squares_gen))  # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

# Generators can be passed directly to functions that consume iterables
total = sum(x**2 for x in range(1000))  # no list created at all!
print(f"Sum of squares: {total}")

# Chaining generator expressions — fully lazy pipeline
numbers = range(1, 101)
evens = (x for x in numbers if x % 2 == 0)
squared = (x**2 for x in evens)
result = sum(squared)   # only computes now
print(f"Sum of squared evens 1-100: {result}")

## 5. Lazy Evaluation: Memory Comparison

In [ ]:
import sys

N = 1_000_000

# List: all values in memory at once
list_of_squares = [x**2 for x in range(N)]
list_size = sys.getsizeof(list_of_squares)

# Generator: only stores current position + frame
gen_of_squares = (x**2 for x in range(N))
gen_size = sys.getsizeof(gen_of_squares)

print(f"List of {N:,} squares: {list_size:,} bytes ({list_size / 1024**2:.1f} MB)")
print(f"Generator:               {gen_size:,} bytes ({gen_size} bytes!)")
print(f"Memory savings: {list_size // gen_size:,}x")

# For sum, both give the same result — generator is always better here
import time

t0 = time.perf_counter()
s = sum([x**2 for x in range(N)])   # builds full list first
t1 = time.perf_counter()
s2 = sum(x**2 for x in range(N))    # purely lazy
t2 = time.perf_counter()

print(f"List approach: {t1-t0:.3f}s")
print(f"Generator approach: {t2-t1:.3f}s")

## 6. Two-Way Communication: `send()` to Generator

`yield` can also **receive** values via `.send(value)`. The `send` call resumes the generator AND sends a value back in, which becomes the return value of the `yield` expression. This turns generators into coroutines.

In [ ]:
def running_average():
    """Coroutine that maintains a running average.
    Send numbers to it; it yields the current average.
    """
    total = 0.0
    count = 0
    average = None

    while True:
        value = yield average   # yield the average OUT, receive value IN
        if value is None:
            break
        total += value
        count += 1
        average = total / count

gen = running_average()
next(gen)   # "prime" the generator — advance to first yield

print(gen.send(10))   # 10.0
print(gen.send(20))   # 15.0
print(gen.send(30))   # 20.0
print(gen.send(40))   # 25.0

gen.close()   # cleanup

## 7. `yield from` — Delegation

`yield from iterable` delegates iteration to another iterable. It's more than syntactic sugar — it also properly forwards `send()` and `throw()` calls through the delegation chain.

In [ ]:
# Without yield from — verbose
def chain_verbose(*iterables):
    for it in iterables:
        for item in it:
            yield item

# With yield from — clean delegation
def chain(* iterables):
    for it in iterables:
        yield from it    # delegates the inner iteration

result = list(chain([1, 2, 3], "ABC", range(4, 7)))
print(result)   # [1, 2, 3, 'A', 'B', 'C', 4, 5, 6]

# Tree traversal using yield from (recursive generators)
def flatten(nested):
    """Recursively flatten an arbitrarily nested list."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # recurse into nested lists
        else:
            yield item

data = [1, [2, [3, 4], 5], [6, 7], 8]
print(list(flatten(data)))   # [1, 2, 3, 4, 5, 6, 7, 8]

## 8. The `itertools` Module

`itertools` provides high-performance, memory-efficient tools for working with iterables. All functions are lazy.

In [ ]:
import itertools

# --- chain: combine iterables ---
print("chain:", list(itertools.chain([1, 2], [3, 4], [5])))

# --- cycle: repeat an iterable forever ---
colors = itertools.cycle(["red", "green", "blue"])
print("cycle:", [next(colors) for _ in range(7)])

# --- islice: lazy slicing ---
def naturals():
    n = 1
    while True:
        yield n
        n += 1

print("islice:", list(itertools.islice(naturals(), 5, 15, 2)))

# --- takewhile / dropwhile ---
print("takewhile:", list(itertools.takewhile(lambda x: x < 5, [1, 2, 3, 4, 5, 6, 1])))
print("dropwhile:", list(itertools.dropwhile(lambda x: x < 5, [1, 2, 3, 4, 5, 6, 1])))

In [ ]:
import itertools

# --- product: Cartesian product ---
suits = ['♠', '♥', '♦', '♣']
ranks = ['A', '2', '3']
cards = list(itertools.product(ranks, suits))
print("product:", cards[:6], "...")

# --- combinations and permutations ---
items = ['a', 'b', 'c', 'd']
print("combinations(4,2):", list(itertools.combinations(items, 2)))
print("permutations(4,2):", list(itertools.permutations(items, 2)))
print("combinations_with_replacement:",
      list(itertools.combinations_with_replacement('AB', 3)))

In [ ]:
import itertools

# --- groupby: group consecutive elements ---
# Important: input must be SORTED by key first!
data = [
    {"name": "Alice", "dept": "Engineering"},
    {"name": "Bob",   "dept": "Engineering"},
    {"name": "Carol", "dept": "Marketing"},
    {"name": "Dave",  "dept": "Marketing"},
    {"name": "Eve",   "dept": "Sales"},
]
data.sort(key=lambda x: x["dept"])

for dept, members in itertools.groupby(data, key=lambda x: x["dept"]):
    names = [m["name"] for m in members]
    print(f"  {dept}: {names}")

## 9. Common Mistakes

In [ ]:
# MISTAKE 1: Generators are single-use — they exhaust!
def my_gen():
    yield 1
    yield 2
    yield 3

gen = my_gen()
print(list(gen))  # [1, 2, 3]
print(list(gen))  # [] -- exhausted! Can't iterate again

# Fix: call the generator function again to get a fresh generator
print(list(my_gen()))  # [1, 2, 3]
print(list(my_gen()))  # [1, 2, 3]

In [ ]:
# MISTAKE 2: Modifying a collection while iterating it
items = [1, 2, 3, 4, 5]

# WRONG: modifying list during iteration causes skipped elements
# for item in items:
#     if item % 2 == 0:
#         items.remove(item)   # BUG: skips elements!

# FIX 1: iterate over a copy
items = [1, 2, 3, 4, 5]
for item in items[:]:    # slice creates a copy
    if item % 2 == 0:
        items.remove(item)
print(items)  # [1, 3, 5]

# FIX 2: use list comprehension (cleaner)
items = [1, 2, 3, 4, 5]
items = [x for x in items if x % 2 != 0]
print(items)  # [1, 3, 5]

In [ ]:
# MISTAKE 3: Not priming a send()-based coroutine
def accumulator():
    total = 0
    while True:
        n = yield total
        total += n

gen = accumulator()
# next(gen)  # MUST prime first (advance to the first yield)

try:
    gen.send(10)   # TypeError: can't send non-None value to a just-started generator
except TypeError as e:
    print(f"TypeError: {e}")

# Fix: always prime first
gen2 = accumulator()
next(gen2)         # prime
print(gen2.send(10))   # 10
print(gen2.send(5))    # 15

## 10. Exercises

**Exercise 1 (Easy):** Write a generator `evens_up_to(n)` that yields all even numbers from 0 to n inclusive.

**Exercise 2 (Medium):** Write a generator `sliding_window(iterable, size)` that yields tuples of consecutive elements. E.g., `sliding_window([1,2,3,4,5], 3)` yields `(1,2,3)`, `(2,3,4)`, `(3,4,5)`.

**Exercise 3 (Hard):** Implement a `pipeline(*funcs)` function that takes a series of generator-producing functions and chains them so data flows through each in order. Use `yield from` and demonstrate with a pipeline: generate numbers → filter evens → square them → take first 5.

In [ ]:
# Exercise 1 Solution
def evens_up_to(n):
    for i in range(0, n + 1, 2):
        yield i

print(list(evens_up_to(10)))

# Exercise 2 Solution
from collections import deque

def sliding_window(iterable, size):
    window = deque(maxlen=size)
    for item in iterable:
        window.append(item)
        if len(window) == size:
            yield tuple(window)

print(list(sliding_window([1, 2, 3, 4, 5], 3)))

In [ ]:
# Exercise 3 Solution: data pipeline
import itertools

def naturals():
    n = 1
    while True:
        yield n
        n += 1

def filter_evens(gen):
    for x in gen:
        if x % 2 == 0:
            yield x

def square(gen):
    for x in gen:
        yield x ** 2

# Build pipeline: naturals -> filter_evens -> square -> take 5
pipeline = itertools.islice(square(filter_evens(naturals())), 5)
print(list(pipeline))   # [4, 16, 36, 64, 100]

## 11. Mini-Project: File Line Reader

A generator-based pipeline for processing large files without loading them into memory. This pattern is used in log analysis, data processing pipelines, and ETL systems.

In [ ]:
import itertools
import io

# Simulate a large file (would normally be an actual file)
SAMPLE_LOG = """
2024-01-01 ERROR: Database connection failed
2024-01-01 INFO: Server started on port 8080
2024-01-01 WARNING: High memory usage detected
2024-01-02 ERROR: Authentication token expired
2024-01-02 INFO: 142 requests processed
2024-01-02 ERROR: Disk write error on /var/log
2024-01-03 INFO: Backup completed successfully
2024-01-03 ERROR: Network timeout after 30s
2024-01-03 WARNING: SSL certificate expires in 7 days
2024-01-04 INFO: Cache cleared
2024-01-04 ERROR: Null pointer dereference in module auth
2024-01-04 INFO: Deployment completed v2.3.1
""".strip()


# --- Generator pipeline components ---

def read_lines(file_obj):
    """Lazily yield lines from a file (never loads whole file into memory)."""
    for line in file_obj:
        yield line.rstrip('\n')


def skip_blanks(lines):
    """Filter out empty lines."""
    for line in lines:
        if line.strip():
            yield line


def filter_by_level(lines, level):
    """Keep only lines with a specific log level."""
    for line in lines:
        if level in line:
            yield line


def parse_log_line(lines):
    """Parse each line into a structured dict."""
    for line in lines:
        parts = line.split(' ', 2)
        if len(parts) == 3:
            date, level_colon, message = parts
            yield {
                "date": date,
                "level": level_colon.rstrip(':'),
                "message": message
            }


def extract_field(records, field):
    """Extract a single field from each dict record."""
    for record in records:
        yield record.get(field, "")


# --- Demo: process the 'file' lazily ---
file_obj = io.StringIO(SAMPLE_LOG)

# Build pipeline
lines    = read_lines(file_obj)
nonempty = skip_blanks(lines)
errors   = filter_by_level(nonempty, "ERROR")
parsed   = parse_log_line(errors)

# Use itertools.islice to peek at first 3 errors
first_3_errors = list(itertools.islice(parsed, 3))

print("First 3 ERROR entries:")
for entry in first_3_errors:
    print(f"  [{entry['date']}] {entry['message']}")

# Reset and run full pipeline — count errors per date
file_obj = io.StringIO(SAMPLE_LOG)
all_errors = list(parse_log_line(filter_by_level(skip_blanks(read_lines(file_obj)), "ERROR")))

print(f"\nTotal ERROR entries: {len(all_errors)}")

# Group by date using itertools.groupby
all_errors.sort(key=lambda x: x["date"])
print("\nErrors per date:")
for date, group in itertools.groupby(all_errors, key=lambda x: x["date"]):
    count = sum(1 for _ in group)
    print(f"  {date}: {count} error(s)")